In [ ]:
# Install dependencies if needed
# !pip install requests beautifulsoup4 pandas ipywidgets

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import time
import random
import ipywidgets as widgets
from IPython.display import display, FileLink

upload_btn = widgets.FileUpload(accept='.csv', multiple=False)
display(upload_btn)

In [ ]:
def read_uploaded_csv(upload_widget):
    import io
    uploaded = list(upload_widget.value.values())
    content = uploaded[0]['content']
    return pd.read_csv(io.BytesIO(content))

data = read_uploaded_csv(upload_btn)
data.head(3)

In [ ]:
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}


def google_search_linkedin_title(full_name: str, company: str, session: requests.Session) -> str:
    """
    Search Google for the person's LinkedIn profile and extract their job title
    from the search result snippet.

    Google snippets for LinkedIn profiles typically look like:
      "Job Title at Company · Experience · ..."
    or the page description contains the title.
    """
    query = f'site:linkedin.com/in "{full_name}" "{company}"'
    url = 'https://www.google.com/search?q=' + quote_plus(query) + '&hl=en'

    try:
        resp = session.get(url, headers=HEADERS, timeout=12)
        resp.raise_for_status()
    except Exception as e:
        return f'[request error: {e}]'

    soup = BeautifulSoup(resp.text, 'html.parser')

    # Each Google organic result lives in a <div class="g"> (or similar).
    # The snippet text is usually inside spans/divs following the title.
    # We look for all text blocks and pick the first one that looks like a
    # LinkedIn snippet (contains "linkedin.com/in" in the parent anchor).

    # Strategy 1: look for the cite tag (shows the URL) then grab the nearby snippet.
    for cite in soup.find_all('cite'):
        if 'linkedin.com/in' in cite.get_text().lower():
            # Walk up to the result container and grab the snippet text
            container = cite.find_parent('div', recursive=True)
            for _ in range(6):          # walk up a few levels
                if container is None:
                    break
                snippet_divs = container.find_all(['div', 'span'])
                for el in snippet_divs:
                    text = el.get_text(separator=' ').strip()
                    # LinkedIn snippets usually mention the person's title before "at" or "·"
                    if len(text) > 20 and ('·' in text or ' at ' in text.lower()):
                        title = _extract_title_from_snippet(text)
                        if title:
                            return title
                container = container.parent

    # Strategy 2: broad sweep of all visible text blocks
    for el in soup.find_all(['span', 'div']):
        text = el.get_text(separator=' ').strip()
        if 40 < len(text) < 400 and ('·' in text or ' at ' in text.lower()):
            title = _extract_title_from_snippet(text)
            if title:
                return title

    return '[title not found]'


def _extract_title_from_snippet(text: str) -> str:
    """
    Pull the job title out of a Google/LinkedIn snippet.

    Common formats:
      "Head of Sales · ACME Corp · 500+ connections"
      "Head of Sales at ACME Corp — LinkedIn"
    """
    # Remove junk prefix tokens Google sometimes adds
    for junk in ['Web result with Site Links', 'More results']:
        text = text.replace(junk, '').strip()

    # Split on middle-dot separator (LinkedIn-style)
    if '·' in text:
        parts = [p.strip() for p in text.split('·')]
        # First part is usually the title; skip if it looks like a name or URL
        candidate = parts[0]
        if candidate and len(candidate) < 120 and 'linkedin' not in candidate.lower():
            return candidate

    # Split on " at " (English-style)
    lower = text.lower()
    if ' at ' in lower:
        idx = lower.index(' at ')
        candidate = text[:idx].strip()
        # Rough sanity check: titles are usually < 80 chars
        if 3 < len(candidate) < 80:
            return candidate

    return ''


print('Functions defined.')

In [ ]:
df = data.copy()

df['Full Name'] = (
    df['First Name'].fillna('') + ' ' + df['Last Name'].fillna('')
).str.strip()

df['LinkedIn Job Title'] = ''
df['Matched LinkedIn URL'] = ''
df['Confidence'] = ''

session = requests.Session()

for i, row in df.iterrows():
    full_name = row['Full Name']
    company   = str(row.get('Outlook Company', '')).strip()

    print(f'[{i+1}/{len(df)}] Looking up: {full_name} @ {company} ...', end=' ')

    title = google_search_linkedin_title(full_name, company, session)
    df.at[i, 'LinkedIn Job Title'] = title
    print(title)

    # Be polite — wait 3-6 seconds between requests to avoid being blocked
    time.sleep(random.uniform(3, 6))

df.head()

In [ ]:
df.to_csv('output_with_titles.csv', index=False)
FileLink('output_with_titles.csv')